<a href="https://colab.research.google.com/github/ProfessorDong/Deep-Learning-Course-Examples/blob/master/ML_Examples/03_Logistic_Regression_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Logistic Regression with PyTorch

1. Design model (input, output size, forward pass)
2. Construct loss and optimizer
3. Training loop
   - forward pass: compute prediction and loss
   - backward pass: gradients
   - update weights

---
**ELC 5365: Deep Learning** | Baylor University

Import torch, numpy, etc.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

Prepare data

In [ ]:
bc = datasets.load_breast_cancer()
X, y = bc.data, bc.target

n_samples, n_features = X.shape
print(f'Number of samples: {n_samples}')
print(f'Number of features: {n_features}')
print(f'Feature names: {bc.feature_names}')
print(f'Class names: {bc.target_names}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)

## Visualize the Input Data

Explore the breast cancer dataset: class distribution, feature distributions, and feature relationships.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Class distribution bar chart
classes, counts = np.unique(y, return_counts=True)
class_names = ['Malignant (0)', 'Benign (1)']
colors = ['#e74c3c', '#2ecc71']
bars = axes[0].bar(class_names, counts, color=colors, edgecolor='black')
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_title('Class Distribution', fontsize=14)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                 str(count), ha='center', fontweight='bold', fontsize=12)

# 2. Feature histogram by class (mean radius)
feature_names = bc.feature_names
for label, name, color in zip([0, 1], ['Malignant', 'Benign'], colors):
    mask = y == label
    axes[1].hist(X[mask, 0], bins=20, alpha=0.6, label=name,
                 color=color, edgecolor='black')
axes[1].set_xlabel(feature_names[0], fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title(f'Distribution of "{feature_names[0]}" by Class', fontsize=14)
axes[1].legend(fontsize=11)

# 3. Scatter plot: mean radius vs mean texture
for label, name, color, marker in zip([0, 1], ['Malignant', 'Benign'],
                                       colors, ['x', 'o']):
    mask = y == label
    axes[2].scatter(X[mask, 0], X[mask, 1], alpha=0.6, label=name,
                    color=color, marker=marker, s=40)
axes[2].set_xlabel(feature_names[0], fontsize=12)
axes[2].set_ylabel(feature_names[1], fontsize=12)
axes[2].set_title(f'{feature_names[0]} vs {feature_names[1]}', fontsize=14)
axes[2].legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of the first 10 features
fig, ax = plt.subplots(figsize=(10, 8))
selected = X[:, :10]
corr_matrix = np.corrcoef(selected.T)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            xticklabels=feature_names[:10],
            yticklabels=feature_names[:10],
            ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Heatmap (First 10 Features)', fontsize=14)
plt.tight_layout()
plt.show()

Scale data

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

y_train = y_train.view(y_train.shape[0], 1)
y_test = y_test.view(y_test.shape[0], 1)

Build model

Linear model $f = wx +b$, sigmoid at the end

In [ ]:
class Model(nn.Module):
    def __init__(self, n_input_features):
        super(Model, self).__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred

model = Model(n_features)

Define loss and optimizer

In [ ]:
learning_rate = 0.01
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

Training loop

In [ ]:
num_epochs = 100
loss_history = []

for epoch in range(num_epochs):
    # Forward pass and loss
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)

    # Backward pass and update
    loss.backward()
    optimizer.step()

    # zero grad before new step
    optimizer.zero_grad()

    loss_history.append(loss.item())

    if (epoch+1) % 10 == 0:
        print(f'epoch: {epoch+1}, loss = {loss.item():.4f}')

## Visualize Training Progress

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), loss_history, 'b-', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('BCE Loss', fontsize=12)
plt.title('Training Loss Over Epochs', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Prediction

In [ ]:
with torch.no_grad():
    y_predicted = model(X_test)
    y_predicted_cls = y_predicted.round()
    acc = y_predicted_cls.eq(y_test).sum() / float(y_test.shape[0])
    print(f'accuracy: {acc.item():.4f}')

## Visualize Results

Evaluate the model with a confusion matrix, predicted probability distribution, and PCA projection of correct vs. incorrect predictions.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.decomposition import PCA

# Get predictions as numpy arrays
y_true = y_test.numpy().flatten()
y_pred_np = y_predicted_cls.numpy().flatten()
y_prob = y_predicted.numpy().flatten()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_true, y_pred_np)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'],
            annot_kws={'size': 16})
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title('Confusion Matrix', fontsize=14)

# 2. Predicted Probability Distribution
axes[1].hist(y_prob[y_true == 0], bins=20, alpha=0.6, color='#e74c3c',
             label='Malignant', edgecolor='black')
axes[1].hist(y_prob[y_true == 1], bins=20, alpha=0.6, color='#2ecc71',
             label='Benign', edgecolor='black')
axes[1].axvline(x=0.5, color='black', linestyle='--', linewidth=2,
                label='Threshold (0.5)')
axes[1].set_xlabel('Predicted Probability', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Predicted Probability Distribution', fontsize=14)
axes[1].legend(fontsize=10)

# 3. PCA scatter: Correct vs Incorrect predictions
pca = PCA(n_components=2)
X_test_2d = pca.fit_transform(X_test.numpy())

correct = (y_pred_np == y_true)
axes[2].scatter(X_test_2d[correct, 0], X_test_2d[correct, 1],
                c='green', marker='o', alpha=0.6, label='Correct', s=50)
axes[2].scatter(X_test_2d[~correct, 0], X_test_2d[~correct, 1],
                c='red', marker='x', alpha=0.8, label='Incorrect', s=80,
                linewidths=2)
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)', fontsize=12)
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)', fontsize=12)
axes[2].set_title('PCA of Test Set (Correct vs Incorrect)', fontsize=14)
axes[2].legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print('Classification Report:')
print('=' * 55)
print(classification_report(y_true, y_pred_np,
                            target_names=['Malignant', 'Benign']))